# Data Exploration - Wilson et al. EEG RSA Replication

This notebook explores the EEG dataset for the Wilson et al. replication study.

## Objectives
- Load and inspect raw EEG data
- Examine data structure and format
- Visualize channel locations and raw signals
- Identify data quality issues
- Generate summary statistics

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from pathlib import Path

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Configuration
DATA_DIR = Path('../data/raw')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

print('Environment setup complete')

## 1. Load Raw EEG Data

Load the raw EEG data files and examine their structure.

In [ ]:
# List available data files
data_files = sorted(DATA_DIR.glob('*.fif'))  # Adjust extension as needed (.set, .vhdr, etc.)
print(f'Found {len(data_files)} data files')

for f in data_files[:5]:  # Show first 5
    print(f'  - {f.name}')

In [ ]:
# Load example dataset
# TODO: Update with actual data path and format
# raw = mne.io.read_raw_fif(data_files[0], preload=True)
# raw = mne.io.read_raw_eeglab(data_files[0], preload=True)
# raw = mne.io.read_raw_brainvision(data_files[0], preload=True)

# For demonstration, create sample data
info = mne.create_info(ch_names=['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2'],
                        sfreq=250,
                        ch_types='eeg')
data = np.random.randn(10, 25000)  # 10 channels, 100 seconds
raw = mne.io.RawArray(data, info)

print(raw)
print(raw.info)

## 2. Examine Recording Information

Inspect key recording parameters and metadata.

In [ ]:
# Display basic information
print(f'Sampling frequency: {raw.info["sfreq"]} Hz')
print(f'Number of channels: {len(raw.ch_names)}')
print(f'Duration: {raw.times[-1]:.2f} seconds')
print(f'\nChannel names: {raw.ch_names}')

In [ ]:
# Get channel types
channel_types = pd.DataFrame({
    'Channel': raw.ch_names,
    'Type': [mne.io.pick.channel_type(raw.info, i) for i in range(len(raw.ch_names))]
})

print('\nChannel type summary:')
print(channel_types['Type'].value_counts())

## 3. Visualize Channel Locations

Display the montage and electrode positions.

In [ ]:
# Set standard montage
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, on_missing='warn')

# Plot sensor locations
fig = raw.plot_sensors(show_names=True, sphere='auto')
plt.tight_layout()

## 4. Visualize Raw EEG Signals

Plot raw EEG data to identify artifacts and data quality.

In [ ]:
# Plot raw data (interactive in Jupyter)
# raw.plot(duration=10, n_channels=10, scalings='auto', block=True)

In [ ]:
# Static plot of first 10 seconds
fig, axes = plt.subplots(5, 1, figsize=(14, 10))
times = raw.times[:2500]  # First 10 seconds at 250 Hz

for i, ax in enumerate(axes):
    data, _ = raw[i, :2500]
    ax.plot(times, data.T * 1e6, linewidth=0.5)  # Convert to µV
    ax.set_ylabel(f'{raw.ch_names[i]}\n(µV)')
    ax.set_xlim(times[0], times[-1])
    if i < len(axes) - 1:
        ax.set_xticklabels([])

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Raw EEG Signals (First 10 seconds)', fontsize=14)
plt.tight_layout()

## 5. Power Spectral Density

Examine frequency content of the signals.

In [ ]:
# Compute and plot PSD
fig = raw.compute_psd(fmax=50).plot(average=True, picks='eeg')
plt.tight_layout()

## 6. Event/Trigger Analysis

Extract and examine experimental events from the data.

In [ ]:
# TODO: Update with actual event extraction method
# events = mne.find_events(raw, stim_channel='STI101')

# For demonstration, create sample events
events = np.array([[i * 500, 0, j % 4 + 1] for i, j in enumerate(range(50))])

print(f'Found {len(events)} events')
print(f'\nEvent codes: {np.unique(events[:, 2])}')

In [ ]:
# Event statistics
event_df = pd.DataFrame({
    'Event Code': events[:, 2],
    'Time (s)': events[:, 0] / raw.info['sfreq']
})

print('Event count by code:')
print(event_df['Event Code'].value_counts().sort_index())

# Visualize event timing
fig = mne.viz.plot_events(events, sfreq=raw.info['sfreq'], first_samp=raw.first_samp)
plt.tight_layout()

## 7. Wilson et al. Specific Analysis

### Experiment Design
TODO: Document the specific experimental paradigm from Wilson et al.
- Stimulus conditions
- Trial structure
- Timing parameters

### Expected Data Structure
TODO: Describe expected event codes and their meanings

In [ ]:
# TODO: Wilson et al. specific event mapping
event_id = {
    'condition_1': 1,
    'condition_2': 2,
    'condition_3': 3,
    'condition_4': 4
}

# Verify all expected conditions are present
observed_codes = set(events[:, 2])
expected_codes = set(event_id.values())

print(f'Expected codes: {expected_codes}')
print(f'Observed codes: {observed_codes}')
print(f'Missing codes: {expected_codes - observed_codes}')

## 8. Data Quality Summary

Generate summary statistics and quality metrics.

In [ ]:
# Summary statistics
data_summary = {
    'Recording Duration (s)': raw.times[-1],
    'Sampling Rate (Hz)': raw.info['sfreq'],
    'Number of Channels': len(raw.ch_names),
    'Number of Events': len(events),
    'Unique Event Codes': len(np.unique(events[:, 2]))
}

summary_df = pd.DataFrame([data_summary]).T
summary_df.columns = ['Value']
print('\n=== Data Summary ===')
print(summary_df)

## Next Steps

1. ✓ Data successfully loaded and explored
2. → Proceed to preprocessing (see `02_preprocessing_demo.ipynb`)
3. → Apply artifact rejection
4. → Epoch data around events
5. → Perform RSA analysis